# Convolutional Neural Network Architecture

This section details the Convolutional Neural Network (CNN) architecture designed for magnetic field prediction. CNNs are specifically suited for processing structured grid data by exploiting spatial relationships through local connectivity and parameter sharing {cite}`lecun1998gradient,krizhevsky2012imagenet,goodfellow2016deep`.

## CNN Fundamentals

Convolutional Neural Networks are similar to ordinary neural networks in that they are made up of learnable weights and biases {cite}`lecun1998gradient,krizhevsky2012imagenet`. However, CNNs make an explicit assumption that **input data follows a structured pattern** to store information (usually in the form of a uniform grid such as images, time-series events, or magnetic field distributions).

This assumption introduces certain properties in the architecture which help in:
1. **Reducing the number of parameters** in the network (critical advantage over fully-connected ANNs)
2. **Making the training process easier** and more efficient through parameter sharing
3. **Exploiting spatial relationships** in the data through local connectivity

:::{important}
**Why CNNs for Electromagnetic Field Prediction**

Magnetic field distributions are naturally represented as 2D spatial grids:
- **Structured data**: Each pixel represents flux density at a specific (x, y) location
- **Spatial correlations**: Maxwell's equations dictate that field at one point depends on nearby regions (∇×B = μJ, ∇·B = 0)
- **Translation equivariance**: Physics laws apply uniformly across the spatial domain
- **Hierarchical patterns**: Local flux gradients → regional saturation zones → global field topology

**Quantitative advantage**: For a 256×256 field distribution:
- Fully-connected network (Section 4a): ~65 million weights in first layer alone
- CNN with 3×3 filters and 64 channels: Only **576 weights** per layer
- **Parameter reduction**: >100,000× fewer parameters while maintaining accuracy
:::

## Main Layer Types in CNN Architecture

There are three main types of layers present in CNN architecture for electromagnetic field prediction:

### 1. Convolutional Layer (CONV)

The convolutional layer computes the connection of each neuron with a local region of the input volume {cite}`lecun1998gradient`. It consists of a set of learnable filters (also called kernels) that extract spatial features.

```{figure} ../_static/figures/FIG2A.pdf
---
name: fig-conv-kernel
width: 60%
---
Convolution kernel visualization. Blue represents the input map, cyan represents the output map. The filter slides across the input, computing dot products.
```

**Key Properties**:
- **Small spatial extent**: Each filter is typically 3×3 or 5×5 (much smaller than input)
- **Sliding window**: Filter slides across entire input with stride (typically 1)
- **Dot product computation**: At each position, compute element-wise multiplication and sum
- **Feature map output**: 2D activation map showing filter response at each spatial location
- **Multiple filters**: Stack of K filters learns K different features, producing K feature maps

**Mathematical Operation**:

For a 2D convolution at position (i,j):
$$
Y[i,j] = \sum_{m} \sum_{n} X[i+m, j+n] \cdot K[m,n] + b
$$

where:
- $X$ is the input (e.g., geometry image or previous layer's feature maps)
- $K$ is the kernel/filter (learnable weights)
- $Y$ is the output feature map
- $b$ is the bias term (one per filter)

**Vectorized form** for entire spatial map:
$$
Y = X \ast K + b
$$

where $\ast$ denotes the convolution operation.

:::{note}
**Electromagnetic Interpretation**

What do convolutional filters learn for magnetic field prediction?

**Early layers** (close to input):
- **Local flux gradients**: Detect sharp changes in B-field magnitude
- **Material boundaries**: Identify transitions between air, iron core, magnets
- **Geometric features**: Recognize edges of slots, poles, magnets

**Middle layers**:
- **Saturation patterns**: Detect regions where core material saturates (nonlinear B-H)
- **Flux paths**: Learn typical flux return paths through iron
- **Interaction zones**: Recognize areas where multiple sources interact

**Deep layers**:
- **Global topology**: Understand overall field distribution symmetry
- **Source localization**: Relate excitation sources to far-field responses
- **Design patterns**: Learn relationships between geometry parameters and field shapes

This hierarchical feature learning mirrors the multi-scale nature of electromagnetic phenomena—from local Ampere's law (∇×H = J) to global flux conservation (∮B·dA = 0).
:::

**Parameter Sharing**:
- Same filter applied at every spatial location
- A 3×3 filter has only 9 weights (plus 1 bias) regardless of input size
- For 64 filters: 64 × (9+1) = **640 parameters** vs. millions for fully-connected layer

### 2. Pooling Layer (POOL)

The pooling layer is responsible for downsampling the spatial dimensions of the input {cite}`lecun1998gradient,goodfellow2016deep`. Pooling layers are periodically inserted in-between convolutional layers to progressively reduce spatial resolution.

**Purpose**:
- **Reduce parameters**: Smaller spatial dimensions → fewer computations in subsequent layers
- **Increase receptive field**: Each neuron "sees" a larger region of the original input
- **Provide translation equivariance**: if the input shifts, the output shifts the same way.
- **Control overfitting**: Reduces model capacity, preventing memorization of training geometries

**Most Popular Form**: Max pooling with $2 \times 2$ filters applied with stride of 2:
- Down-samples every depth slice by 2 along both width and height
- **Dimension reduction**: H×W → H/2×W/2 (75% spatial reduction)
- **Channel preservation**: Depth dimension remains unchanged
- **Operation**: $Y[i,j] = \max(X[2i:2i+2, 2j:2j+2])$ for each channel

**Hyperparameters**:
1. **Filter Size (F)**: Typically $2 \times 2$ or $3 \times 3$
2. **Stride (S)**: Step size for sliding the filter (typically 2 for max pooling)

**Output dimensions**:
$$
H_{\text{out}} = \lfloor (H_{\text{in}} - F) / S \rfloor + 1
$$
$$
W_{\text{out}} = \lfloor (W_{\text{in}} - F) / S \rfloor + 1
$$

**Example**: A $4 \times 4$ input with $2 \times 2$ max pooling (stride 2) produces a $2 \times 2$ output:

```
Input (4×4): Output (2×2) after max pooling:
[1 3 | 2 4] [3 | 4]
[5 6 | 1 2] [6 | 2]
-------+------ ---+---
[7 2 | 8 3] [7 | 8]
[1 0 | 4 1] [1 | 4]

Takes max of each 2×2 block
```

:::{note}
**Trade-offs for Electromagnetic Field Prediction**

**Advantages**:
- **Computational efficiency**: Training and inference speed increases significantly
- **Receptive field growth**: Deeper layers can "see" larger portions of the motor geometry
- **Abstraction**: Forces network to learn high-level electromagnetic patterns

**Disadvantages**:
- **Spatial information loss**: Precise flux density locations become blurred
- **Boundary details**: Sharp transitions (air-iron interfaces) may be smeared
- **Resolution reduction**: Difficult to predict fine-grained field variations

**Solution in this work**: The encoder-decoder architecture with **skip connections** (Section below) recovers lost spatial information during upsampling, combining coarse semantic features from encoder with fine spatial details from early layers.
:::

### 3. Fully Connected Layer (FC)

Neurons in fully connected layers have dense connections with all activations in the previous layer, similar to regular neural networks.

**Differences from CONV Layers**:
- **Structure**: CONV layers exhibit local connectivity and parameter sharing
- **Function**: Both compute dot products between weights and activations, followed by non-linear activation

**Purpose**: 
- Balance the locality-context tradeoff
- Relate all features learned by CONV layers
- Develop patterns that capture global trends

**Note**: In modern encoder-decoder architectures for dense prediction (like field estimation), FC layers are often omitted in favor of fully convolutional architectures.

## Dilated Convolutions

A key improvement in the magnetic field prediction architecture is the use of **dilated convolutions** (also called atrous convolutions) {cite}`yu2015multi`. This technique is critical for capturing long-range electromagnetic interactions without increasing computational cost.

### Standard vs. Dilated Convolution

**Standard Convolution**: Dense kernel captures only immediate neighbors

```{figure} ../_static/figures/FIG2A.pdf
---
name: fig-standard-conv
width: 45%
---
Non-dilated convolutional filter with padding. All kernel elements are contiguous.
```

**Dilated Convolution**: Sparse kernel captures distant relationships

```{figure} ../_static/figures/FIG2B.pdf
---
name: fig-dilated-conv
width: 45%
---
Dilated filter with dilation rate of 1. Kernel elements are spaced apart, expanding the receptive field without increasing parameters.
```

### Mathematical Formulation

**Standard convolution**:
$$
Y[i,j] = \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} X[i+m, j+n] \cdot K[m,n]
$$

**Dilated convolution** with dilation rate $r$:
$$
Y[i,j] = \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} X[i+r \cdot m, j+r \cdot n] \cdot K[m,n]
$$

**Receptive field**: Effective filter size becomes $(k-1) \cdot r + 1$
- $r=1$ (standard): 3×3 filter has receptive field of 3×3 = 9 pixels
- $r=2$ (dilation=1): 3×3 filter has receptive field of 5×5 = 25 pixels
- $r=3$ (dilation=2): 3×3 filter has receptive field of 7×7 = 49 pixels

**Key advantage**: Exponential receptive field growth without parameter increase!

### Motivation for Magnetic Field Prediction

**Physical Insight from Electromagnetic Theory**:

The magnetic field at a point is determined by the **Biot-Savart law**:
$$
\mathbf{B}(\mathbf{r}) = \frac{\mu_0}{4\pi} \int \frac{\mathbf{J}(\mathbf{r'}) \times (\mathbf{r} - \mathbf{r'})}{|\mathbf{r} - \mathbf{r'}|^3} dV'
$$

This shows that **the field at location $\mathbf{r}$ depends on current density $\mathbf{J}$ at ALL locations $\mathbf{r'}$**—including spatially distant regions. The $1/|\mathbf{r} - \mathbf{r'}|^2$ falloff means distant sources still contribute significantly.

**Practical implications**:
- **Airgap flux** in a motor is influenced by rotor magnets, stator windings, AND iron saturation in the back iron (cm away)
- **Fringing fields** extend far from slot openings
- **Mutual coupling** between coils separated by multiple slots
- **Leakage flux paths** through air can span the entire motor diameter

**Why standard 3×3 convolutions are insufficient**:
- With stride=1 and no pooling, receptive field after $N$ layers is only $(2N+1) \times (2N+1)$
- For 10 layers: receptive field = 21×21 pixels
- For a 256×256 motor image: Only sees 8% of spatial dimension
- To see entire motor would need ~128 layers (infeasible to train)

**Why dilated convolutions solve this** {cite}`yu2015multi`:
- Alternating dilated layers exponentially expand receptive field
- After 10 layers with alternating dilation [1,2,1,2,...]: receptive field ~200×200 pixels
- Sees ~78% of spatial domain with 10× fewer layers
- **Same parameter count** as standard convolutions (still 3×3 filters)

:::{important}
**Empirical Results from This Work**

Ablation studies (Section 4d) demonstrate:

| Architecture | Test NMSE | Receptive Field |
|--------------|-----------|-----------------|
| Standard CNN (no dilation) | 1.2% | ~60×60 pixels |
| CNN with dilated layers | **0.4%** | ~200×200 pixels |
| Improvement | **3× better** | **11× larger** |

**Conclusion**: Dilated convolutions are **essential** for accurate electromagnetic field prediction. Standard CNNs cannot capture the long-range interactions dictated by Maxwell's equations without prohibitively deep architectures.
:::

### Implementation Details

In this work:
- **Dilation pattern**: Alternating [1, 2, 1, 2, ...] in encoder layers
- **Kernel size**: 3×3 (compact even with dilation)
- **Placement**: Applied in encoder layers 2, 4, 6 (after initial feature extraction)
- **Decoder**: Standard convolutions (dilation not needed for reconstruction)

## Encoder-Decoder Architecture

The deep learning model for magnetic field prediction uses an **encoder-decoder architecture** {cite}`ronneberger2015unet`, similar to the U-Net architecture designed for medical image segmentation but adapted for dense regression instead of classification.

```{figure} ../_static/figures/Network_arch.png
---
name: fig-network-arch
width: 95%
---
Encoder-Decoder based Convolutional Neural Network Architecture for magnetic field prediction. The encoder (left half) progressively downsamples to extract abstract features. The decoder (right half) upsamples to reconstruct the spatial field distribution. Skip connections (horizontal arrows) preserve fine-grained details.
```

### Encoder Section (Contracting Path)

The encoder extracts hierarchical features from the input geometry representation:

**Progressive abstraction** (left side of network):
1. **Layer 1-2** (256×256 → 128×128): Local geometric features
- Slot edges, magnet boundaries, winding positions
- Learns: "Where are the materials located?"

2. **Layer 3-4** (128×128 → 64×64): Regional electromagnetic patterns
- Flux concentration zones, saturation regions
- Learns: "Where will flux be high/low?"

3. **Layer 5-6** (64×64 → 32×32): Global field topology
- Flux return paths, symmetry patterns
- Learns: "What is the overall field structure?"

4. **Bottleneck** (32×32): Compressed latent representation
- Most discriminative features for this geometry
- Learns: "What are the key electromagnetic relationships?"

**Technical components per encoder block**:
- Two $3 \times 3$ convolutional layers (some with dilation)
- ReLU activation after each convolution {cite}`nair2010rectified`
- Batch normalization for stable training {cite}`ioffe2015batch`
- $2 \times 2$ max-pooling with stride 2 (75% spatial reduction)
- Dropout (rate=0.5) for regularization {cite}`srivastava2014dropout`

**Feature channel growth**: 32 → 64 → 128 → 256 as spatial dimensions shrink
- Compensates for spatial information loss with richer feature representations
- Total information capacity remains approximately constant

### Decoder Section (Expanding Path)

The decoder reconstructs the full-resolution field distribution from the compressed latent representation:

**Progressive reconstruction** (right side of network):
1. **Bottleneck → Layer 7-8** (32×32 → 64×64): Coarse field structure
- Major flux paths, high-level topology
- Reconstructs: "Overall field shape"

2. **Layer 9-10** (64×64 → 128×128): Regional field details
- Saturation zone boundaries, flux gradients
- Reconstructs: "Regional variations"

3. **Layer 11-12** (128×128 → 256×256): Fine-grained field distribution
- Precise flux density at each spatial location
- Reconstructs: "Pixel-level predictions"

**Technical components per decoder block**:
- $2 \times 2$ transposed convolution (upsampling) with stride 2
- Concatenation with corresponding encoder feature maps (skip connections)
- Two $3 \times 3$ convolutional layers
- ReLU activation and batch normalization
- Dropout for regularization

**Feature channel reduction**: 256 → 128 → 64 → 32 → 1 as spatial dimensions grow
- Progressively focuses on spatial precision over feature diversity
- Final layer: 1 channel = predicted flux density magnitude

### Skip Connections (Critical COmponent)

Skip connections directly connect encoder layers to corresponding decoder layers {cite}`ronneberger2015unet`:

**Why skip connections are essential**:

**Problem**: Spatial information is lost during encoder downsampling
- Max-pooling discards 75% of spatial positions at each step
- After 4 pooling operations: 256×256 → 16×16 (99.6% spatial reduction)
- Decoder must reconstruct field from highly compressed representation
- **Result without skip connections**: Blurry predictions, poor boundary localization

**Solution**: Concatenate encoder feature maps with decoder feature maps
```
Decoder Layer 9 input = [Upsampled Layer 8] ⊕ [Encoder Layer 3]
(low-res features) (high-res features)
```

where ⊕ denotes channel-wise concatenation.

**Benefits**:
1. **Precise localization**: Encoder features preserve exact positions of geometric boundaries
2. **Sharp transitions**: Air-iron interfaces, magnet edges remain crisp
3. **Gradient flow**: Direct paths from output to early layers improve training
4. **Multi-scale fusion**: Combines semantic (what) and spatial (where) information

:::{tip}
**Electromagnetic Context**

Skip connections are particularly important for magnetic field prediction because:

- **Material boundaries**: Sharp flux density transitions at air-iron interfaces (B changes by 1000×) must be precisely localized
- **Slot openings**: Fringing fields have strong spatial gradients requiring accurate positioning
- **Airgap flux**: Small airgap (1-2mm) needs sub-pixel accuracy to predict torque correctly
- **Saturation zones**: Exact spatial extent of core saturation affects overall performance

**Quantitative impact** (ablation study, Section 4d):
- With skip connections: 0.4% NMSE, sharp boundaries
- Without skip connections: 2.1% NMSE, blurry boundaries
- **5× accuracy improvement** from skip connections alone!
:::

### Comparison to U-Net

This architecture is inspired by U-Net {cite}`ronneberger2015unet` but adapted for electromagnetic field prediction:

| Aspect | U-Net (Medical Imaging) | This Work (EM Fields) |
|--------|------------------------|----------------------|
| **Task** | Segmentation (classification) | Regression (continuous values) |
| **Output** | Class labels per pixel | Flux density per pixel |
| **Output activation** | Softmax | Linear (no activation) |
| **Loss function** | Cross-entropy | Mean Squared Error |
| **Innovation added** | - | **Dilated convolutions** for long-range EM interactions |
| **Receptive field** | Standard | **Exponentially expanded** via dilation |

**Key insight**: U-Net's architecture is well-suited for any dense prediction task requiring both semantic understanding (encoder) and precise localization (decoder + skip connections).

## Network Specifications

The complete deep learning model consists of:

| Component | Count |
|-----------|-------|
| **Total Layers** | 32 |
| **Trainable Convolutional Layers** | 16 |
| **Pooling/Up-sampling Layers** | 8 |
| **Dropout Layers** | 8 |
| **Total Parameters** | 2.4 Million |

### Layer Configuration

Each encoder/decoder block consists of:
1. Two sets of $3 \times 3$ convolution layers
2. ReLU activation after each convolution
3. Batch normalization layer {cite}`ioffe2015batch`
4. $2 \times 2$ max-pooling (encoder) or up-sampling (decoder) with stride 2
5. Dropout for regularization {cite}`srivastava2014dropout` (typically 0.5 dropout rate)

**Exception**: The last two convolutional layers do not follow this pattern due to a network design constraint to fit the field distribution dimensionality.

### Convolution Details

- **Kernel Size**: $3 \times 3$ (with some $5 \times 5$)
- **Stride**: 1 for all convolutional layers
- **Number of Filters (K)**: Varies by layer, typically K=32 or K=64
- **Dilation**: Alternate dilated layers in encoder with dilation rate of 1

**Key Architectural Characteristics for Electromagnetic Field Prediction**:
- **Encoder-decoder structure** {cite}`ronneberger2015unet` for dense spatial regression
- **Dilated convolutions** {cite}`yu2015multi` to capture long-range electromagnetic interactions
- **Skip connections** to preserve fine-grained flux density details
- **Batch normalization** {cite}`ioffe2015batch` for stable training on FEA-generated data

## Model Capacity and Generalization

For the network to train successfully, it must have **sufficient capacity** to satisfactorily capture the complexity of the electromagnetic field prediction problem. This capability is achieved by choosing appropriate values for {cite}`goodfellow2016deep`:

1. **Number of Layers**: Depth of the network (32 total layers)
2. **Kernel Size**: Size of convolutional filters ($3 \times 3$, $5 \times 5$)
3. **Number of Kernels**: Filters per layer (K=32, K=64)
4. **Regularization Parameters**: Dropout rate, batch normalization

### Model Selection Process

Model selection is performed by {cite}`bergstra2012random`:
- Investigating a set of candidate models
- Choosing the model with best balance between fit and complexity
- Using random search and grid search for hyperparameter optimization

**Result**: A total of 35 networks with different configurations were trained during the model selection process before arriving at the final architecture described here.

:::{note}
**Understanding This Visualization** 
This demonstration shows how a convolutional filter processes input data:

1. **Input (5×5)**: Could represent a small patch of a magnetic field or geometry image
2. **Kernel (3×3)**: Learnable filter that detects specific patterns (e.g., edges, gradients)
3. **Feature Map (3×3)**: Output showing where the pattern is detected (high values = strong match)

The red box on the input shows the **receptive field**—the region that influences one output pixel. The filter slides across the entire input, computing the dot product at each position.

**For electromagnetic field prediction**: Early-layer filters learn to detect geometric features (slot edges, magnet boundaries), while deeper filters learn electromagnetic patterns (flux concentration zones, saturation regions).
:::

In [ ]:
# CNN core-ops visualization (refactored: cleaner, bigger pixels, less clutter)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def cnn_core_ops_viz(seed=42):
    rng = np.random.default_rng(seed)

    # ----- Data (small, readable) -----
    X = rng.random((5, 5))
    X[1:4, 1:4] += 0.6  # add a structured "hot" region (like a flux gradient)

    # Simple Sobel-like vertical edge kernel
    K = np.array([[-1, 0,  1],
                  [-2, 0,  2],
                  [-1, 0,  1]], dtype=float) / 4.0

    # Valid convolution output: (5-3+1)=3
    Y = np.zeros((3, 3), dtype=float)
    for i in range(3):
        for j in range(3):
            patch = X[i:i+3, j:j+3]
            Y[i, j] = np.sum(patch * K)

    # Pick a demonstration location for the "one pixel" breakdown
    demo_i, demo_j = 0, 0
    patch = X[demo_i:demo_i+3, demo_j:demo_j+3]
    demo_val = np.sum(patch * K)

    # Larger input for receptive field demo
    Z = rng.random((7, 7))
    Z[2:5, 2:5] += 0.6

    # ----- Plot helpers -----
    def show_mat(ax, A, title, cmap=None, annotate=False, fmt="{:.2f}", vmin=None, vmax=None):
        im = ax.imshow(A, cmap=cmap, interpolation="nearest", vmin=vmin, vmax=vmax)
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_aspect("equal")
        # NO gridlines (they create “mini pixels”)
        if annotate:
            for r in range(A.shape[0]):
                for c in range(A.shape[1]):
                    ax.text(c, r, fmt.format(A[r, c]),
                            ha="center", va="center",
                            fontsize=10, fontweight="bold",
                            color="white" if (cmap in ["RdBu", "coolwarm"] and abs(A[r, c]) > 0.2) else "black")
        return im

    # ----- Figure layout -----
    fig = plt.figure(figsize=(16, 10), constrained_layout=True)
    gs = fig.add_gridspec(3, 3, height_ratios=[1.0, 0.65, 1.0])

    # Top row: input, kernel, output
    ax_in  = fig.add_subplot(gs[0, 0])
    ax_k   = fig.add_subplot(gs[0, 1])
    ax_out = fig.add_subplot(gs[0, 2])

    im_in = show_mat(ax_in, X, "Input feature map (5×5)\n(e.g., flux density / geometry)", cmap="viridis", annotate=False)
    # highlight the receptive field used for output[0,0]
    ax_in.add_patch(Rectangle((demo_j - 0.5, demo_i - 0.5), 3, 3,
                              fill=False, edgecolor="red", linewidth=3))
    # ax_in.text(2, -0.9, "Receptive field for output[0,0]",
    #            ha="center", fontsize=9, color="red", fontweight="bold")
    
    ax_in.text(
        0.5, -0.05,
        "Receptive field for output[0,0]",
        transform=ax_in.transAxes,
        ha="center", va="top",
        fontsize=9, color="red", fontweight="bold"
    )



    # Kernel (annotate values)
    im_k = show_mat(ax_k, K, "Kernel (3×3)\n(detects vertical gradients)", cmap="RdBu",
                    annotate=True, vmin=-0.5, vmax=0.5)

    # Output (annotate values)
    im_out = show_mat(ax_out, Y, "Output feature map (3×3)\n(edge response)", cmap="plasma",
                      annotate=True)

    # One shared colorbar per row (cleaner than 3 separate colorbars)
    cbar1 = fig.colorbar(im_in, ax=ax_in, fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=9)

    cbar2 = fig.colorbar(im_k, ax=ax_k, fraction=0.046, pad=0.02)
    cbar2.ax.tick_params(labelsize=9)

    cbar3 = fig.colorbar(im_out, ax=ax_out, fraction=0.046, pad=0.02)
    cbar3.ax.tick_params(labelsize=9)

    # Middle row: step-by-step calculation for output[0,0]
    ax_mid = fig.add_subplot(gs[1, :])
    ax_mid.axis("off")

    patch_str = "\n".join(["  " + " ".join([f"{patch[r,c]:5.2f}" for c in range(3)]) for r in range(3)])
    kern_str  = "\n".join(["  " + " ".join([f"{K[r,c]:5.2f}"     for c in range(3)]) for r in range(3)])

    text = (
        f"Computing output[{demo_i},{demo_j}]\n\n"
        f"1) Take the 3×3 input patch\n{patch_str}\n\n"
        f"2) Multiply element-wise by the 3×3 kernel\n{kern_str}\n\n"
        f"3) Sum all products →  output[{demo_i},{demo_j}] = {demo_val:.4f}"
    )
    ax_mid.text(0.5, 0.5, text, ha="center", va="center",
                fontsize=11, fontfamily="monospace",
                bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.55),
                transform=ax_mid.transAxes)

    # Bottom row: standard vs dilated receptive field
    ax_std = fig.add_subplot(gs[2, 0])
    ax_dil = fig.add_subplot(gs[2, 1])
    ax_txt = fig.add_subplot(gs[2, 2])
    ax_txt.axis("off")

    show_mat(ax_std, Z, "Standard 3×3\nReceptive field = 3×3", cmap="viridis")
    ax_std.add_patch(Rectangle((2 - 0.5, 2 - 0.5), 3, 3, fill=False, edgecolor="red", linewidth=3))
    for r in range(2, 5):
        for c in range(2, 5):
            ax_std.plot(c, r, "r*", markersize=12)

    show_mat(ax_dil, Z, "Dilated 3×3 (dilation=2)\nReceptive field = 5×5", cmap="viridis")
    ax_dil.add_patch(Rectangle((1 - 0.5, 1 - 0.5), 5, 5, fill=False, edgecolor="dodgerblue", linewidth=3))
    # dilation=2 means taps at (1,1), (1,3), (1,5), ...
    taps = [(1,1),(1,3),(1,5),
            (3,1),(3,3),(3,5),
            (5,1),(5,3),(5,5)]
    for r, c in taps:
        ax_dil.plot(c, r, "*", color="dodgerblue", markersize=12)

    ax_txt.text(
        0.5, 0.5,
        "Key idea:\n\n"
        "• Standard conv samples a contiguous 3×3 neighborhood\n"
        "• Dilated conv samples a wider area (5×5 here)\n"
        "• Same 3×3 weights → same parameter count\n\n"
        "Why it helps EM fields:\n"
        "• Captures longer-range interactions (e.g., across slots)\n"
        "• Improves context without deeper networks",
        ha="center", va="center", fontsize=11,
        bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.7),
        transform=ax_txt.transAxes
    )

    fig.suptitle("Convolutional Neural Network: Core Operations", fontsize=16, fontweight="bold")
    plt.show()

cnn_core_ops_viz()


## Summary

This section showed how convolutional neural networks can be structured to act as **efficient surrogate solvers for electromagnetic field problems**.

Rather than treating field data as unstructured vectors, the CNN architecture explicitly exploits the **spatial organization of magnetic fields**. Local convolutions capture sharp gradients and material interfaces, while deeper layers aggregate these features into region-level and global field patterns. This mirrors the multi-scale nature of electromagnetic phenomena, from local flux crowding to global return paths.

A key architectural choice is the use of **dilated convolutions**, which expand the network’s receptive field without increasing parameter count. This allows the model to account for long-range electromagnetic interactions—an essential requirement given that magnetic fields depend on distant sources, not only local geometry.

Downsampling in the encoder provides context and efficiency, while **skip connections** ensure that spatial precision is retained during reconstruction. Together, these elements enable dense, high-resolution field prediction without the prohibitive cost of fully connected networks or repeated finite-element solves.

The result is a compact yet expressive model—on the order of a few million parameters—that can predict full 2D magnetic field distributions in milliseconds. This makes CNNs practical tools for design exploration, optimization, and uncertainty analysis, where thousands of field evaluations are required.

The next section (4c) focuses on how this architecture is trained on FEA-generated datasets and how convergence, generalization, and error behavior are assessed in practice.
